## To Shift Scale and Combine the Images

In [1]:
# importing all the relevant libraries 
import numpy as np
import astropy
import astroalign as aa
import photutils
import ccdproc
import os
import astropy.io.fits as fits
import matplotlib.pyplot as plt
from ccdproc import CCDData, combiner
from astropy import units as u
from astropy.time import Time
from matplotlib.colors import LogNorm
from photutils.centroids import centroid_com, centroid_2dg, centroid_sources
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.segmentation import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift
import gc
gc.enable()

In [2]:
bands = ["b", "g", "r"]

# positions of bright unsaturated stars
#                 p1x,p1y,      p2x, p2y,   p3x, p3y
positions_all = [[(603.44,1027), (807,863.08), (499.24, 855.88)], # b band pos
                 [(603.44,1027), (807,863.08), (499.24, 855.88)],  
                 [(603.44,1027), (807,863.08), (499.24, 855.88)]
                 ]   #x-y notation
apertures_all = [15.0 , 0.0, 0.0]

aa_refidk = [5, 0, 0]
sc_refidk = [0, 0, 0]

# Navigating to folder that contains all the image files - only run this once
# If this block has already run, uncomment the next line and run again
# os.chdir("..")
os.chdir("NGC_2547_sub/")

In [ ]:
# Band selector
#   [b, g, r]
#   [0, 1, 2]
fidx = 0

# Assignments
positions = positions_all[fidx]
ap_r = apertures_all[fidx]


In [7]:
print('*_'+bands[fidx] + '.fit')

*_b.fit


In [ ]:
# Collating b-band images
images = ccdproc.ImageFileCollection(".", glob_include='*_b.fit')  # collect all b-band images
#for fn in images.files_filtered():  # loop through and print all b-band files
    #print(fn)

scim = []                                                                    # This is an empty list - scim
for fn in images.files_filtered():
    print(fn)
    scim.append(CCDData.read(fn, unit = "adu"))                              # reads the file and adds it to the list 
    # This code will adds MJD-OBS to the header so it will (later) stop producing error messages
    times = scim[-1].header['DATE-OBS']                                      # Get the time
    t = Time(times, format='isot', scale='utc')                              # Read the time from UTC format
    scim[-1].header['MJD-OBS'] = (float(t.mjd), 'MJD') 

## Shifting
#### Using the automated alignment to shift all images to the correct orientation 

In [ ]:
# generating new names for the automatically aligned images
newname=[]                                                              # creating a new empty variable
for fn in images.files_filtered():                                      # iterating through all the images
    newname.extend(["aa"+fn])                                           # creating a list of file names with the original filenames and adding "aa" infront
print(newname)                                                          # printing the new names

In [ ]:
# Automated alignment using image no6 as the reference image
refidx = 5

# save aligned images in this folder
os.chdir("aligned/")

for idx, thisimage in enumerate(scim):
    # Define images to be shifted
    # Plus zero is a quirky fix to an endian issue (old school)
    img_ref=scim[refidx].data+0       # Reference image
    img_example=scim[idx].data+0      # Image to be shifted image
    # print('Reference and example image defined')

    img_aligned, footprint = aa.register(img_example, img_ref, detection_sigma=5.0, min_area=9, fill_value=-99999.99)
    # print('Image alignment determined')

    temp=scim[idx]
    temp.data=img_aligned
    temp.meta = scim[idx].meta

    # Write the image to a file using the new name (aa- prefix)
    print(newname[idx])
    temp.write(newname[idx])


command prompt lines

ds9 aaLight_NGC* &


# Scaling

In [ ]:
# Collating b-band images
images = ccdproc.ImageFileCollection(".", glob_include='*_b.fit')  # collect all b-band images
#for fn in images.files_filtered():  # loop through and print all b-band files
    #print(fn)

scim = []                                                                    # This is an empty list - scim
for fn in images.files_filtered():
    print(fn)
    scim.append(CCDData.read(fn, unit = "adu"))                              # reads the file and adds it to the list 
    # This code will adds MJD-OBS to the header so it will (later) stop producing error messages
    times = scim[-1].header['DATE-OBS']                                      # Get the time
    t = Time(times, format='isot', scale='utc')                              # Read the time from UTC format
    scim[-1].header['MJD-OBS'] = (float(t.mjd), 'MJD') 


# Creating photometric table and apertures
# Three positions of bright unsaturated stars
positions = positions_all[fidx]
apertures = CircularAperture(positions, r=ap_r)
phot_table = aperture_photometry(scim[0], apertures)
#print(phot_table)
phot_tables=[]
for idx, thisimage in enumerate(scim):                                  # Comment - summing the fluxes of the apertures of each image 
    phot_tables.extend([aperture_photometry(thisimage, apertures)])      
    #print(idx, phot_tables[idx])


refidx=0 # The idex of our reference image. By default it is zero, but should it be different?

# Combine

In [ ]:
# This code is missing comments. What is it doing?
#images = ccdproc.ImageFileCollection(".",glob_include='aaLight_*')                      # collecting the sproc images 
#scim = [CCDData.read(fn) for fn in images.files_filtered()]                                 # saving the image data in scim

# go back one folder
os.chdir("..")

for idx, thisimage in enumerate(scim):                                                          # iterating through each image        
    m = np.ma.median(phot_tables[refidx]['aperture_sum'] / phot_tables[idx]['aperture_sum'])    # scaling each image with the reference image
    print(m)
    scim[idx] = scim[idx].multiply(m * u.adu)                                       # normalising adu counts with reference image

# go back one folder
os.chdir("..")
# Comment on the combine operations, including why minmax_clip_min = -500    
sci_average = ccdproc.combine(scim, method = 'average',dtype = np.float32,          # mean combining images  
                              minmax_clip = True, minmax_clip_min = -500)           # rejecting any data with -500 adu counts in the average combining
sci_average.write("NGC_2547_b_average.fits", overwrite=True)

sci_median = ccdproc.combine(scim, method = 'median',dtype = np.float32,            # median combining images
                             minmax_clip = True, minmax_clip_min = -500)
sci_median.write("NGC_2547_b_median.fits", overwrite=True)

del(scim)
collected = gc.collect()
print('Check garbage collection', collected)